# 📤 Load — Silver & Gold → Supabase

Os arquivos parquet são ótimos para análise local, mas a API precisa de algo consultável online. Vamos subir o que importa para o Postgres do Supabase:

- `contracts` ← `silver/contracts.parquet`
- `payments` ← `silver/payments.parquet`
- `contract_features` ← `gold/contract_features.parquet`

Antes de rodar este notebook, certifique-se que rodou `sql/001_schema.sql` no Supabase e que `DATABASE_URL` está no seu `.env`.

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path.cwd().parents[1]
DATA    = PROJECT / 'notebooks' / 'data'

import pandas as pd
from sqlalchemy import create_engine, text, MetaData
from sqlalchemy.dialects.postgresql import insert
from dotenv import load_dotenv

load_dotenv(PROJECT / '.env')
DATABASE_URL = os.environ['DATABASE_URL']

engine = create_engine(DATABASE_URL, pool_pre_ping=True)

## Sanity check de conexão

In [ ]:
with engine.connect() as conn:
    db = conn.execute(text('select current_database()')).scalar()
    tables = conn.execute(text(
        "select tablename from pg_tables where schemaname = 'public' order by tablename"
    )).scalars().all()

print('DB:', db)
print('Tabelas:', tables)

Se a lista de tabelas estiver vazia, rode `sql/001_schema.sql` antes de prosseguir.

## Helper de upsert

Idempotente: `INSERT ... ON CONFLICT DO UPDATE`. Pode rodar o notebook quantas vezes quiser.

In [ ]:
def upsert(df: pd.DataFrame, table_name: str, pk: str, chunk: int = 1000) -> int:
    if df.empty:
        return 0
    meta = MetaData()
    meta.reflect(bind=engine, only=[table_name])
    tbl = meta.tables[table_name]
    cols = {c.name for c in tbl.columns}

    df2 = df[[c for c in df.columns if c in cols]].copy()
    records = df2.where(pd.notna(df2), None).to_dict(orient='records')

    total = 0
    with engine.begin() as conn:
        for i in range(0, len(records), chunk):
            batch = records[i:i + chunk]
            stmt = insert(tbl).values(batch)
            update_cols = {c.name: c for c in stmt.excluded if c.name != pk}
            stmt = stmt.on_conflict_do_update(index_elements=[pk], set_=update_cols)
            conn.execute(stmt)
            total += len(batch)
    return total

## Carga

In [ ]:
started = datetime.now(timezone.utc)

contracts = pd.read_parquet(DATA / 'silver' / 'contracts.parquet')
n_c = upsert(contracts, 'contracts', pk='id_contrato')
print(f'contracts: {n_c}')

In [ ]:
payments = pd.read_parquet(DATA / 'silver' / 'payments.parquet')
n_p = upsert(payments, 'payments', pk='id_pagamento', chunk=2000)
print(f'payments: {n_p}')

In [ ]:
features = pd.read_parquet(DATA / 'gold' / 'contract_features.parquet')
n_f = upsert(features, 'contract_features', pk='id_contrato')
print(f'contract_features: {n_f}')

## Auditoria

In [ ]:
ended = datetime.now(timezone.utc)
with engine.begin() as conn:
    conn.execute(
        text("""
            INSERT INTO pipeline_runs (run_type, status, started_at, ended_at, rows_in, rows_out)
            VALUES (:rt, :st, :sa, :ea, :ri, :ro)
        """),
        {
            'rt': 'full_load', 'st': 'success',
            'sa': started, 'ea': ended,
            'ri': n_c + n_p, 'ro': n_c + n_p + n_f,
        },
    )
print('Duração:', ended - started)

## Smoke test

In [ ]:
with engine.connect() as c:
    for q in [
        'SELECT COUNT(*) AS n FROM contracts',
        'SELECT COUNT(*) AS n FROM payments',
        'SELECT COUNT(*) AS n FROM contract_features',
        """SELECT status_cobranca, COUNT(*) AS n
             FROM contracts GROUP BY 1 ORDER BY n DESC""",
    ]:
        print('>>', q.strip().split('\n')[0])
        for row in c.execute(text(q)):
            print('  ', dict(row._mapping))